In [1]:
"""
Step 1: Find and Catalogue Multi-Sector TESS Targets
=======================================================
 
WHY THIS STEP EXISTS:
The cross-sector consistency idea (Section 3.6) only works on stars that were
observed by TESS more than once. A single-sector observation gives us no way
to distinguish a real astrophysical signal from instrumental noise -- we need
independent repeat looks at the same star to check if an anomaly is
consistent (real) or one-off (likely noise/artifact).
 
We prioritize the Continuous Viewing Zone (CVZ) near the ecliptic poles
because TESS's sector geometry means CVZ stars can get 20+ sectors of
coverage (vs. 1-2 for most stars), giving us much stronger statistical power
per star for the consistency check.
 
Output: a CSV of TIC IDs with >= MIN_SECTORS sectors of data, sorted by
sector count. This becomes the input for Step 2 (light curve download).
"""

"\nStep 1: Find and Catalogue Multi-Sector TESS Targets\n=======================================================\n\nWHY THIS STEP EXISTS:\nThe cross-sector consistency idea (Section 3.6) only works on stars that were\nobserved by TESS more than once. A single-sector observation gives us no way\nto distinguish a real astrophysical signal from instrumental noise -- we need\nindependent repeat looks at the same star to check if an anomaly is\nconsistent (real) or one-off (likely noise/artifact).\n\nWe prioritize the Continuous Viewing Zone (CVZ) near the ecliptic poles\nbecause TESS's sector geometry means CVZ stars can get 20+ sectors of\ncoverage (vs. 1-2 for most stars), giving us much stronger statistical power\nper star for the consistency check.\n\nOutput: a CSV of TIC IDs with >= MIN_SECTORS sectors of data, sorted by\nsector count. This becomes the input for Step 2 (light curve download).\n"

In [3]:
from astroquery.mast import Observations
import numpy as np
import pandas as pd

In [4]:
# Minimum number of independent sectors required for a star to be useful
# for cross-sector consistency checking. 3 is a reasonable floor -- below
# this, "consistency" is statistically too weak to claim much.

MIN_SECTORS = 3

In [5]:
# Where the final target list gets saved. This becomes the backbone that
# every later pipeline step (light curve download, VAE, IF/LOF) reads from.

OUTPUT_CSV = "multisector_tic_targets.csv"

In [6]:
# Approximate CVZ regions, as (RA, Dec) box centers + a search radius.
# The ecliptic poles sit at roughly RA=90, Dec=+/-66.56 -- the 66.56 comes
# from 90 - 23.44 (Earth's axial tilt), since the ecliptic pole is offset
# from the celestial pole by exactly the tilt angle.
#
# WHY A BOX AND NOT THE WHOLE SKY: pushing this coordinate filter into the
# query itself (rather than downloading everything and filtering later)
# keeps the MAST query fast and keeps us from downloading metadata for
# thousands of irrelevant single-sector stars we'd just throw away anyway.

In [7]:
CVZ_REGIONS = [
    {"name":"north_cvz","ra" : 90.0,"dec" : 66.56,"radius_deg":12},
    {"name":"south_cvz","ra" : 90.0,"dec" : -66.56,"radius_deg":12}
]

In [8]:
# =========================================================================
# STEP FUNCTIONS
# =========================================================================

In [9]:
def query_cvz_region(ra,dec,radius_deg):
    """
    Query MAST for TESS timeseries observations near a given (ra, dec).
 
    WHY obs_collection="TESS": restricts results to the TESS mission only --
    MAST hosts data from many missions (Hubble, JWST, Kepler, etc.) and we
    don't want those mixed in.
 
    WHY dataproduct_type="timeseries": TESS observations come in several
    product types (full-frame images, target pixel files, light curve
    timeseries, ...). We only need light curve metadata right now (actual
    files get downloaded in Step 2), so filtering to "timeseries" avoids
    pulling back a lot of irrelevant image-product rows and keeps the query
    fast.
 
    WHY s_ra / s_dec as ranges: this is a simple bounding-box search. It's
    not a perfectly accurate cone search (RA lines converge near the poles,
    so a naive box gets slightly distorted there), but it's good enough for
    building an initial candidate list -- we're not making final scientific
    claims off this step, just picking targets.
    """
    obs = Observations.query_criteria(
        obs_collection="TESS",
        dataproduct_type="timeseries",
        s_ra=[ra-radius_deg,ra+radius_deg],
        s_dec=[dec-radius_deg,dec+radius_deg]
    )
    # Convert from astropy Table to pandas DataFrame -- groupby/filtering/
    # sorting are all much easier in pandas than in an astropy Table.

    return obs.to_pandas()

In [10]:
def extract_tic_id(target_name):
    """
    Pull a clean, digits-only TIC ID out of MAST's target_name field.
 
    WHY THIS IS NECESSARY: MAST's target_name column is inconsistently
    formatted -- sometimes "TIC 123456789", sometimes just "123456789",
    sometimes a non-TIC proposal-specific name. If we grouped directly on
    the raw string, "TIC 123456789" and "123456789" would be treated as two
    DIFFERENT stars (string mismatch), silently under-counting that star's
    real sector count. Non-TIC names would also pollute the final target
    list with garbage entries. This function is a defensive parsing step
    that catches both problems before they become invisible bugs downstream.
    """

    if pd.isna(target_name):
        return None
    s = str(target_name).replace("TIC","").strip()
    return s if s.isdigit() else None

In [11]:
# =========================================================================
# STEP A: Query both CVZ regions and combine into one observation-level table
# =========================================================================
#
# At this point, obs_df has ONE ROW PER SECTOR-OBSERVATION, not per star.
# If a star was observed in 5 sectors, it appears as 5 separate rows here.
# This distinction matters for Step B below.

In [12]:
all_obs = []

for region in CVZ_REGIONS:
    print(f"Querying {region["name"]}...")
    df = query_cvz_region(region["ra"],region["dec"],region["radius_deg"])

    # Tag which CVZ region each row came from -- useful for later debugging
    # or analysis (e.g. "are north vs south CVZ giving different results?").

    df["cvz_region"] = region["name"]
    all_obs.append(df)
    print(f"  -> {len(df)} observation rows")

# ignore_index=True re-numbers rows 0,1,2,... after stacking -- without it,
# the two DataFrames' original indices could overlap/clash.

obs_df = pd.concat(all_obs,ignore_index=True)

Querying north_cvz...
  -> 7588 observation rows
Querying south_cvz...
  -> 133198 observation rows


In [13]:
obs_df

,intentType,obs_collection,provenance_name,instrument_name,project,filters,wave_region,target_name,target_classification,obs_id,...,jpegURL,dataURL,dataRights,mtFlag,srcDen,obsid,objID,wave_min,wave_max,cvz_region
0,science,TESS,SPOC,Photometer,TESS,TESS,Optical,137985716,NaN,tess2019199201929-s0014-s0041-0000000137985716,...,NaN,mast:TESS/product/tess2019199201929-s0014-s004...,PUBLIC,False,NaN,64265490,120242009,600.0,1000.0,north_cvz
1,science,TESS,SPOC,Photometer,TESS,TESS,Optical,138042556,NaN,tess2019199201929-s0014-s0041-0000000138042556,...,NaN,mast:TESS/product/tess2019199201929-s0014-s004...,PUBLIC,False,NaN,64265491,120242014,600.0,1000.0,north_cvz
2,science,TESS,SPOC,Photometer,TESS,TESS,Optical,138168262,NaN,tess2019199201929-s0014-s0041-0000000138168262,...,NaN,mast:TESS/product/tess2019199201929-s0014-s004...,PUBLIC,False,NaN,64265492,120242015,600.0,1000.0,north_cvz
3,science,TESS,SPOC,Photometer,TESS,TESS,Optical,138168780,NaN,tess2019199201929-s0014-s0041-0000000138168780,...,NaN,mast:TESS/product/tess2019199201929-s0014-s004...,PUBLIC,False,NaN,64265494,120242021,600.0,1000.0,north_cvz
4,science,TESS,SPOC,Photometer,TESS,TESS,Optical,138171285,NaN,tess2019199201929-s0014-s0041-0000000138171285,...,NaN,mast:TESS/product/tess2019199201929-s0014-s004...,PUBLIC,False,NaN,64265498,120242025,600.0,1000.0,north_cvz
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140781,science,TESS,SPOC,Photometer,TESS,TESS,OPTICAL,257720712,NaN,tess2018206190142-s0001-s0096-0000000257720712,...,NaN,mast:TESS/product/tess2018206190142-s0001-s009...,PUBLIC,False,NaN,403515688,1127387259,600.0,1000.0,south_cvz
140782,science,TESS,SPOC,Photometer,TESS,TESS,OPTICAL,257720712,NaN,tess2025232030459-s0096-0000000257720712-0293-s,...,NaN,mast:TESS/product/tess2025232030459-s0096-0000...,PUBLIC,False,NaN,354017662,1127387275,600.0,1000.0,south_cvz
140783,science,TESS,SPOC,Photometer,TESS,TESS,OPTICAL,382256692,NaN,tess2025232030459-s0096-0000000382256692-0293-s,...,NaN,mast:TESS/product/tess2025232030459-s0096-0000...,PUBLIC,False,NaN,354028414,1127436740,600.0,1000.0,south_cvz
140784,science,TESS,SPOC,Photometer,TESS,TESS,OPTICAL,382256692,NaN,tess2018206190142-s0001-s0096-0000000382256692,...,NaN,mast:TESS/product/tess2018206190142-s0001-s009...,PUBLIC,False,NaN,403560085,1127436758,600.0,1000.0,south_cvz


In [ ]:
# =========================================================================
# STEP B: Extract clean TIC IDs, then collapse observation-level rows down
#          to one row per star with a count of its distinct sectors
# =========================================================================